# WSI CZI Processing Pipeline

**Pipeline order:**
1. **Cell 1** – Configuration (edit all paths/settings here)
2. **Step A-1** – Per-slide RGB stats CSV *(crash-safe: resumes from last completed slide)*
3. **Step A-2** – Merge CSVs → pixel-rule pre-filter → SVM classification
4. **Step QC** – Visual patch inspection (`VISUALIZE_PATCHES` flag controls this)
5. **Step B** – Patch extraction helper (used inline during Step C)
6. **Step C** – H-Optimus-1 feature extraction → `.pt` files per patch

---
**Filtration pipeline (Step A-2):**
Patches are excluded in two stages:
1. **Pixel rules** (fast, no model needed) — any of these fires → patch excluded immediately:
   - `black_pixel_ratio > BLACK_PIXEL_RATIO` — dark edge/fold artifact
   - `white_pixel_ratio > WHITE_PIXEL_RATIO` — bright background
   - Mean R/G/B all > `WHITE_MEAN_THRESH` **and** std R/G/B all < `WHITE_STD_THRESH` — uniform pale glass
2. **SVM** — runs on patches that passed the pixel rules; labels remaining white vs tissue

---
**Output folder structure:**
```
OUTPUT_ROOT/
  patch_metadata/                  ← per-slide RGB CSVs (Step A-1)
  patch_metadata_merged.csv        ← all patches with white_label (Step A-2)
  patch_metadata_nonwhite.csv      ← tissue-only rows (Step A-2)
  qc_patches/
    white/                         ← QC: sample excluded patches
    nonwhite/                      ← QC: sample tissue patches
    filtered_by_rule/              ← patches rejected by pixel rule (pre-SVM)
      rule_filtered_patches.csv
      black_pixel_ratio/
      white_pixel_ratio/
      white_mean_std/
  features/
    SR1482_40X_HE_T004_02/         ← .pt feature files per patch (Step C)
    ...
```


## Cell 1 — Configuration (edit everything here)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
import os

# ── Input paths ────────────────────────────────────────────────────────────────
CZI_ROOT          = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline"
SR386_LABELS_CSV  = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline\SR386_labels.csv"
SR1482_LABELS_CSV = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline\SR1482_labels.csv"
SVM_MODEL_PATH    = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline\tcga_svm_model.pkl"

# ── Output path ────────────────────────────────────────────────────────────────
OUTPUT_ROOT = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline\Output\V5"

# ── Patch settings ─────────────────────────────────────────────────────────────
PATCH_SIZE = 512   # pixels at target magnification
TARGET_MAG = 20    # desired magnification (x)

# ── Feature extraction ─────────────────────────────────────────────────────────
HF_TOKEN = os.environ.get("HF_TOKEN", "")
DEVICE   = "cuda"   # "cuda" or "cpu"

# ── Pixel filter thresholds (Stage 1 of Step A-2) ─────────────────────────────
BLACK_PIXEL_RATIO = 0.20   # >20% black pixels  → edge/fold artifact
WHITE_PIXEL_RATIO = 0.90   # >90% white pixels  → background
WHITE_MEAN_THRESH = 210    # mean R/G/B all above → likely glass background
WHITE_STD_THRESH  = 12     # std  R/G/B all below → no tissue texture

# ── Flags ──────────────────────────────────────────────────────────────────────
# Set to 1 to save QC PNG patches (original behaviour).
# Set to 0 to skip Step QC entirely — no PNGs written, much faster overall.
VISUALIZE_PATCHES = 1

# How many patch-rows to read per CZI call in Step A-1 (row-strip batching).
# Each strip = ROW_BATCH * PATCH_SIZE native pixels tall.
# Lower to 4 or 8 if you see memory warnings.
ROW_BATCH = 16

# Number of patches per GPU forward pass in Step C.
# Increase if GPU has spare VRAM; decrease if you get OOM errors.
FEATURE_BATCH_SIZE = 8

# ── Derived output paths (do not edit) ────────────────────────────────────────
PATCH_METADATA_DIR    = os.path.join(OUTPUT_ROOT, "patch_metadata")
MERGED_METADATA_CSV   = os.path.join(OUTPUT_ROOT, "patch_metadata_merged.csv")
NONWHITE_METADATA_CSV = os.path.join(OUTPUT_ROOT, "patch_metadata_nonwhite.csv")
FEATURES_ROOT         = os.path.join(OUTPUT_ROOT, "features")
QC_FILTERED_DIR       = os.path.join(OUTPUT_ROOT, "qc_patches", "filtered_by_rule")

for _d in [PATCH_METADATA_DIR, FEATURES_ROOT, QC_FILTERED_DIR]:
    os.makedirs(_d, exist_ok=True)

print("Configuration loaded.")
print(f"  CZI root         : {CZI_ROOT}")
print(f"  Output root      : {OUTPUT_ROOT}")
print(f"  Target mag       : {TARGET_MAG}x  |  Patch size: {PATCH_SIZE}")
print(f"  SVM model        : {SVM_MODEL_PATH}")
print(f"  Visualize patches: {'ON' if VISUALIZE_PATCHES else 'OFF'}")
print(f"  Row batch        : {ROW_BATCH} rows per CZI read")
print(f"  Feature batch    : {FEATURE_BATCH_SIZE} patches/GPU call")
print(f"  Black filter     : >{BLACK_PIXEL_RATIO:.0%} black pixels")
print(f"  White filter     : >{WHITE_PIXEL_RATIO:.0%} white pixels  |  mean>{WHITE_MEAN_THRESH} & std<{WHITE_STD_THRESH}")


Configuration loaded.
  CZI root         : D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline
  Output root      : D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline\Output\V5
  Target mag       : 20x  |  Patch size: 512
  SVM model        : D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline\tcga_svm_model.pkl
  Visualize patches: ON
  Row batch        : 16 rows per CZI read
  Feature batch    : 8 patches/GPU call
  Black filter     : >20% black pixels
  White filter     : >90% white pixels  |  mean>210 & std<12


## Step A-1 — Helpers: label lookup & CZI reader

In [2]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

# ── Label DataFrames loaded once at first call ────────────────────────────────
_sr386_labels  = None
_sr1482_labels = None

def _load_label_dfs():
    global _sr386_labels, _sr1482_labels
    if _sr386_labels is None:
        _sr386_labels = pd.read_csv(SR386_LABELS_CSV)
        _sr386_labels['case_id'] = _sr386_labels['case_id'].astype(str).str.zfill(3)
        print(f"  Loaded SR386 labels: {len(_sr386_labels)} rows")
    if _sr1482_labels is None:
        _sr1482_labels = pd.read_csv(SR1482_LABELS_CSV)
        _sr1482_labels['case_id'] = _sr1482_labels['case_id'].astype(str).str.zfill(3)
        print(f"  Loaded SR1482 labels: {len(_sr1482_labels)} rows")


def get_slide_label(czi_filename: str) -> str | None:
    _load_label_dfs()
    stem = Path(czi_filename).stem

    match = re.search(r'_T(\d{3})_', stem)
    if not match:
        print(f"  [SKIP] {stem}: could not extract case ID from filename")
        return None
    case_id = match.group(1)

    if stem.upper().startswith("SR386"):
        row = _sr386_labels[_sr386_labels['case_id'] == case_id]
        if row.empty:
            print(f"  [SKIP] {stem}: case_id '{case_id}' not found in SR386 labels CSV")
            return None
        val = row.iloc[0]['mmr_loss_binary']
        if val == 1:   return 'msih'
        elif val == 0: return 'nonmsih'
        print(f"  [SKIP] {stem}: unknown mmr_loss_binary value '{val}'")
        return None

    elif stem.upper().startswith("SR1482"):
        row = _sr1482_labels[_sr1482_labels['case_id'] == case_id]
        if row.empty:
            print(f"  [SKIP] {stem}: case_id '{case_id}' not found in SR1482 labels CSV")
            return None
        val = str(row.iloc[0]['MSI']).strip()
        if val.lower() == 'msi high':          return 'msih'
        elif val.lower() == 'no msi':          return 'nonmsih'
        elif val.lower() in ('not performed', 'insufficient', 'failed'):
            print(f"  [SKIP] {stem}: MSI test not usable ('{val}')"); return None
        elif val.lower() == 'msi low':
            print(f"  [SKIP] {stem}: MSI Low — excluded per protocol"); return None
        print(f"  [SKIP] {stem}: unknown MSI value '{val}'")
        return None

    print(f"  [SKIP] {stem}: filename does not match SR386 or SR1482 prefix")
    return None


def find_all_czi_files(root: str) -> list[str]:
    """Recursively find all .czi files under root."""
    results = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.lower().endswith('.czi'):
                results.append(os.path.join(dirpath, f))
    return sorted(results)


print("Helpers loaded.")


Helpers loaded.


## Step A-1 — CZI reader with magnification handling

In [3]:
from pylibCZIrw import czi as pyczi

def open_czi_at_target_mag(czi_path: str, target_mag: int = 20, patch_size: int = 512):
    with pyczi.open_czi(czi_path) as czidoc:
        try:
            native_mag = int(czidoc.metadata['ImageDocument']['Metadata']
                             ['Information']['Instrument']['Objectives']
                             ['Objective']['NominalMagnification'])
        except Exception:
            native_mag = 40

        bbox    = czidoc.total_bounding_box
        X_start = bbox['X'][0]
        Y_start = bbox['Y'][0]
        W       = bbox['X'][1] - bbox['X'][0]
        H       = bbox['Y'][1] - bbox['Y'][0]

        print(f"    bbox raw: X={bbox['X']}, Y={bbox['Y']}")

        downsample = max(1, native_mag // target_mag)
        level_mag  = native_mag // downsample

    return {
        'czi_path'  : czi_path,
        'W'         : W, 'H': H,
        'X_start'   : X_start, 'Y_start': Y_start,
        'native_mag': native_mag,
        'level_mag' : level_mag,
        'downsample': downsample,
    }


## Step A-1 — Generate per-slide RGB stats CSV (row-strip batching)

> **Memory-safe row-strip approach:**
> Instead of reading the whole slide (3–8 GB) or one patch at a time (slow),
> the code reads `ROW_BATCH` rows of patches at once as a single horizontal strip.
> Stats are computed with **vectorized NumPy** on the strip — no Python loops
> over individual patches. The strip is discarded before the next is read,
> so peak RAM = one strip, not the whole slide.

> **Crash-safe resume behaviour:**
> - Each slide writes to a `.tmp` file patch-strip by patch-strip
> - Only when a slide is **fully done** is the `.tmp` renamed to the final `.csv`
> - If the program crashes mid-slide, the stale `.tmp` is removed on the next run and that slide restarts from the beginning
> - Slides whose final `.csv` already exists are **skipped entirely**
> - At most one slide's worth of work is lost on any crash


In [4]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm


def compute_patch_rgb_stats_rowbatch(slide_info: dict, patch_size: int,
                                      row_batch: int = 16) -> pd.DataFrame:
    """
    Row-strip batched RGB stats — memory-safe for multi-GB slides.

    For each group of `row_batch` patch-rows, reads one horizontal strip,
    reshapes into individual patches, and computes all stats with vectorized
    NumPy. The strip is released before the next one is read.
    """
    from pylibCZIrw import czi as pyczi

    H, W       = slide_info['H'], slide_info['W']
    downsample = slide_info['downsample']
    czi_path   = slide_info['czi_path']
    x_start    = slide_info['X_start']
    y_start    = slide_info['Y_start']
    zoom       = 1.0 / downsample

    H_target = H // downsample
    W_target = W // downsample

    n_patch_rows = H_target // patch_size
    n_patch_cols = W_target // patch_size

    if n_patch_rows == 0 or n_patch_cols == 0:
        return pd.DataFrame()

    all_rows  = []
    patch_num = 0

    with pyczi.open_czi(czi_path) as czidoc:
        for strip_start_row in tqdm(range(0, n_patch_rows, row_batch),
                                     desc="    strips", unit="strip", leave=False):
            strip_end_row   = min(strip_start_row + row_batch, n_patch_rows)
            n_rows_in_strip = strip_end_row - strip_start_row

            y_native_strip = y_start + (strip_start_row * patch_size * downsample)
            height_native  = n_rows_in_strip * patch_size * downsample
            width_native   = n_patch_cols * patch_size * downsample
            x_native_strip = x_start

            strip = czidoc.read(
                roi=(x_native_strip, y_native_strip, width_native, height_native),
                zoom=zoom,
                plane={'C': 0, 'Z': 0, 'T': 0},
            )[:, :, :3]

            strip_H = n_rows_in_strip * patch_size
            strip_W = n_patch_cols    * patch_size
            strip   = strip[:strip_H, :strip_W, :]   # guard against CZI rounding

            strip_f32 = strip.astype(np.float32)
            grid = strip_f32.reshape(n_rows_in_strip, patch_size,
                                     n_patch_cols,    patch_size, 3)
            grid    = grid.transpose(0, 2, 1, 3, 4)
            N       = n_rows_in_strip * n_patch_cols
            patches = grid.reshape(N, patch_size, patch_size, 3)

            avg = patches.mean(axis=(1, 2))
            std = patches.std(axis=(1, 2))

            p8         = patches.astype(np.uint8)
            black_mask = (p8[:,:,:,0] < 15)  & (p8[:,:,:,1] < 15)  & (p8[:,:,:,2] < 15)
            white_mask = (p8[:,:,:,0] > 210) & (p8[:,:,:,1] > 210) & (p8[:,:,:,2] > 210)

            total_px          = patch_size * patch_size
            black_pixel_ratio = black_mask.sum(axis=(1, 2)) / total_px
            white_pixel_ratio = white_mask.sum(axis=(1, 2)) / total_px

            row_idx, col_idx = np.unravel_index(np.arange(N), (n_rows_in_strip, n_patch_cols))
            abs_row_idx = row_idx + strip_start_row
            patch_x     = col_idx     * patch_size
            patch_y     = abs_row_idx * patch_size

            for i in range(N):
                all_rows.append({
                    'patch_number'     : patch_num + i,
                    'patch_x'          : int(patch_x[i]),
                    'patch_y'          : int(patch_y[i]),
                    'avg_R'            : float(avg[i, 0]),
                    'avg_G'            : float(avg[i, 1]),
                    'avg_B'            : float(avg[i, 2]),
                    'std_R'            : float(std[i, 0]),
                    'std_G'            : float(std[i, 1]),
                    'std_B'            : float(std[i, 2]),
                    'black_pixel_ratio': float(black_pixel_ratio[i]),
                    'white_pixel_ratio': float(white_pixel_ratio[i]),
                })
            patch_num += N

            del strip, strip_f32, grid, patches, p8, black_mask, white_mask

    return pd.DataFrame(all_rows)


def run_step_A1(czi_root, output_metadata_dir, patch_size, target_mag, row_batch):
    """
    Generate per-slide RGB stats CSVs with crash-safe resume logic.

    Resume behaviour
    ----------------
    Each slide writes to a  <stem>.tmp  file first.
    Only when fully done is it renamed to  <stem>.csv.
    On the next run:
      - <stem>.csv exists  → slide skipped (already complete)
      - <stem>.tmp exists  → stale from a crash; deleted and slide restarts
    At most one slide is ever re-processed after a crash.
    """
    czi_files = find_all_czi_files(czi_root)
    print(f"Found {len(czi_files)} CZI file(s) under {czi_root}")

    for czi_path in tqdm(czi_files, desc="Step A1: RGB stats", unit="slide"):
        filename = os.path.basename(czi_path)
        stem     = Path(filename).stem

        label = get_slide_label(filename)
        if label is None:
            continue

        final_csv = os.path.join(output_metadata_dir, f"{stem}.csv")
        tmp_csv   = os.path.join(output_metadata_dir, f"{stem}.tmp")

        # ── Already complete ───────────────────────────────────────────────
        if os.path.exists(final_csv):
            tqdm.write(f"  [SKIP] {stem}: CSV already exists")
            continue

        # ── Stale temp file from a previous crash ─────────────────────────
        if os.path.exists(tmp_csv):
            tqdm.write(f"  [RESUME] {stem}: removing stale .tmp from previous run, restarting")
            os.remove(tmp_csv)

        tqdm.write(f"  Processing {stem}  (label={label}) ...")
        try:
            slide_info = open_czi_at_target_mag(czi_path, target_mag, patch_size)
        except Exception as e:
            tqdm.write(f"  [ERROR] Could not open {stem}: {e}")
            continue

        tqdm.write(f"    native={slide_info['native_mag']}x  "
                   f"target size=({slide_info['W']//slide_info['downsample']} x "
                   f"{slide_info['H']//slide_info['downsample']})  "
                   f"downsample={slide_info['downsample']}")

        try:
            df = compute_patch_rgb_stats_rowbatch(slide_info, patch_size, row_batch)
        except Exception as e:
            tqdm.write(f"  [ERROR] Stats failed for {stem}: {e}")
            # Clean up any partial tmp file if it somehow got written
            if os.path.exists(tmp_csv):
                os.remove(tmp_csv)
            continue

        if df.empty:
            tqdm.write(f"  [WARN] {stem}: no patches extracted (slide too small?)")
            continue

        df.insert(0, 'slide_name', stem)
        df.insert(1, 'label',      label)

        # ── Write to tmp, then atomically rename to final ──────────────────
        df.to_csv(tmp_csv, index=False)
        os.rename(tmp_csv, final_csv)
        tqdm.write(f"    → {len(df):,} patches | saved to {final_csv}")


run_step_A1(CZI_ROOT, PATCH_METADATA_DIR, PATCH_SIZE, TARGET_MAG, ROW_BATCH)


Found 2 CZI file(s) under D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline


Step A1: RGB stats: 100%|██████████| 2/2 [00:00<00:00, 142.86slide/s]

  Loaded SR386 labels: 427 rows
  Loaded SR1482 labels: 416 rows
  [SKIP] SR1482_40X_HE_T004_02: CSV already exists
  [SKIP] SR386_40X_HE_T014_01: CSV already exists


## Step A-2 — Merge CSVs → pixel-rule pre-filter → SVM classification

In [5]:
import joblib
import glob
import numpy as np
import pandas as pd


def apply_svm_whiteness(svm_input_df: pd.DataFrame, svm_model_path: str) -> pd.DataFrame:
    """
    Run the trained SVM on patches that passed the pixel-rule pre-filter.
    Returns the input DataFrame with a 'white_label' column added.

    The SVM was trained on features: ['avg_G', 'avg_B', 'std_R'].
    white_label = 1  → white/background
    white_label = 0  → tissue
    """
    svm_bundle = joblib.load(svm_model_path)

    if isinstance(svm_bundle, dict):
        svm_model = svm_bundle['model']
        scaler    = svm_bundle.get('scaler', None)
    elif isinstance(svm_bundle, (list, tuple)) and len(svm_bundle) == 2:
        svm_model, scaler = svm_bundle
    else:
        svm_model = svm_bundle
        scaler    = None

    required_features = ['avg_G', 'avg_B', 'std_R']
    for col in required_features:
        if col not in svm_input_df.columns:
            raise ValueError(f"Missing required feature column for SVM: '{col}'")

    X = svm_input_df[required_features].values
    if scaler is not None:
        X = scaler.transform(X)

    y_pred    = svm_model.predict(X)
    result_df = svm_input_df.copy()
    result_df['white_label'] = y_pred

    unique_vals = np.unique(y_pred)
    if len(unique_vals) != 2:
        print(f"  WARNING: SVM predicted only one class: {unique_vals}")

    white_class = max(unique_vals)
    n_white     = (y_pred == white_class).sum()
    n_tissue    = (y_pred != white_class).sum()
    total       = len(y_pred)
    print(f"  SVM results  — white: {n_white:,} ({100*n_white/total:.1f}%)  "
          f"tissue: {n_tissue:,} ({100*n_tissue/total:.1f}%)")

    return result_df


print("apply_svm_whiteness loaded.")


apply_svm_whiteness loaded.


## Step A-2 — Run: merge → pixel pre-filter → SVM → save CSVs

In [6]:
import os
import glob
import pandas as pd
import numpy as np


def tag_bad_patches(df: pd.DataFrame):
    """
    Stage 1: apply three pixel-based rules to flag obvious white/background patches.
    Patches flagged here are labelled white immediately and skipped by the SVM.

    Returns (df_with_labels, bad_mask).
    """
    if 'black_pixel_ratio' not in df.columns or 'white_pixel_ratio' not in df.columns:
        raise ValueError(
            "[ERROR] 'black_pixel_ratio' / 'white_pixel_ratio' columns missing.\n"
            "  → Re-run Step A-1 to regenerate per-slide CSVs."
        )

    is_black       = df['black_pixel_ratio'] > BLACK_PIXEL_RATIO
    is_white_ratio = df['white_pixel_ratio'] > WHITE_PIXEL_RATIO
    is_white_mean  = (
        (df['avg_R'] > WHITE_MEAN_THRESH) & (df['avg_G'] > WHITE_MEAN_THRESH) &
        (df['avg_B'] > WHITE_MEAN_THRESH) & (df['std_R'] < WHITE_STD_THRESH)  &
        (df['std_G'] < WHITE_STD_THRESH)  & (df['std_B'] < WHITE_STD_THRESH)
    )
    bad = is_black | is_white_ratio | is_white_mean

    df = df.copy()
    df['white_label']   = float('nan')
    df['reject_reason'] = ''
    df.loc[bad, 'white_label']             = 1
    df.loc[is_black,       'reject_reason'] = 'black_pixel_ratio'
    df.loc[is_white_ratio, 'reject_reason'] = 'white_pixel_ratio'
    df.loc[is_white_mean,  'reject_reason'] = 'white_mean_std'

    print(f"  Stage 1 — pixel-rule pre-filter:")
    print(f"    black_pixel_ratio  (>{BLACK_PIXEL_RATIO:.0%}) : {is_black.sum():,}")
    print(f"    white_pixel_ratio  (>{WHITE_PIXEL_RATIO:.0%}) : {is_white_ratio.sum():,}")
    print(f"    white_mean_std  fallback           : {is_white_mean.sum():,}")
    print(f"    Total force-labelled white         : {bad.sum():,}")
    print(f"    Passed to SVM (Stage 2)            : {(~bad).sum():,}")
    return df, bad


def run_step_A2(patch_metadata_dir, merged_csv_path, nonwhite_csv_path, svm_model_path):
    # ── 1. Merge all per-slide CSVs ──────────────────────────────────────────
    csv_files = sorted(glob.glob(os.path.join(patch_metadata_dir, '*.csv')))
    if not csv_files:
        raise FileNotFoundError(
            f"[ERROR] No per-slide CSVs found in: {patch_metadata_dir}\n"
            f"  → Run Step A-1 first."
        )

    print(f"Merging {len(csv_files)} per-slide CSV(s) ...")
    dfs = []
    for csv_path in csv_files:
        try:
            dfs.append(pd.read_csv(csv_path))
        except Exception as e:
            print(f"  [WARN] Could not read {csv_path}: {e} — skipping")

    if not dfs:
        raise RuntimeError("[ERROR] All per-slide CSVs failed to load.")

    merged_df = pd.concat(dfs, ignore_index=True)
    print(f"  → {len(merged_df):,} total patches from {len(dfs)} slide(s)")

    # ── 2. Stage 1: pixel-rule pre-filter ───────────────────────────────────
    merged_df, bad_mask = tag_bad_patches(merged_df)

    filtered_df = merged_df[bad_mask][[
        'slide_name', 'patch_number', 'patch_x', 'patch_y', 'reject_reason',
        'black_pixel_ratio', 'white_pixel_ratio',
        'avg_R', 'avg_G', 'avg_B', 'std_R', 'std_G', 'std_B'
    ]].copy()
    filtered_csv = os.path.join(QC_FILTERED_DIR, "rule_filtered_patches.csv")
    filtered_df.to_csv(filtered_csv, index=False)
    print(f"  Pixel-rule rejected patch list → {filtered_csv}")

    # ── 3. Stage 2: SVM on patches that passed the pixel rules ──────────────
    svm_input_df = merged_df[merged_df['white_label'].isna()].drop(
        columns=['white_label', 'reject_reason']
    )
    print(f"\n  Stage 2 — SVM on {len(svm_input_df):,} patches ...")
    svm_result_df = apply_svm_whiteness(svm_input_df, svm_model_path)

    # ── 4. Merge SVM labels back into main DataFrame ─────────────────────────
    merged_df.loc[merged_df['white_label'].isna(), 'white_label'] = \
        svm_result_df['white_label'].values
    merged_df['white_label'] = merged_df['white_label'].astype(int)

    # ── 5. Save merged CSV (all patches + white_label) ───────────────────────
    save_df = merged_df.drop(columns=['reject_reason'])
    save_df.to_csv(merged_csv_path, index=False)

    # ── 6. Save nonwhite CSV (tissue patches only) ───────────────────────────
    nonwhite_class = save_df['white_label'].min()
    nonwhite_df    = save_df[save_df['white_label'] == nonwhite_class].reset_index(drop=True)
    nonwhite_df.to_csv(nonwhite_csv_path, index=False)

    total    = len(save_df)
    n_white  = (save_df['white_label'] != nonwhite_class).sum()
    n_tissue = len(nonwhite_df)

    print(f"\nStep A-2 complete.")
    print(f"  Total patches      : {total:,}")
    print(f"  White (excluded)   : {n_white:,}  ({100*n_white/total:.1f}%)")
    print(f"  Tissue (kept)      : {n_tissue:,}  ({100*n_tissue/total:.1f}%)")
    print(f"  Merged CSV         → {merged_csv_path}")
    print(f"  Non-white CSV      → {nonwhite_csv_path}")
    return nonwhite_df


run_step_A2(
    patch_metadata_dir = PATCH_METADATA_DIR,
    merged_csv_path    = MERGED_METADATA_CSV,
    nonwhite_csv_path  = NONWHITE_METADATA_CSV,
    svm_model_path     = SVM_MODEL_PATH,
)


Merging 2 per-slide CSV(s) ...
  → 34,435 total patches from 2 slide(s)
  Stage 1 — pixel-rule pre-filter:
    black_pixel_ratio  (>20%) : 18,628
    white_pixel_ratio  (>90%) : 1,978
    white_mean_std  fallback           : 1,451
    Total force-labelled white         : 20,636
    Passed to SVM (Stage 2)            : 13,799
  Pixel-rule rejected patch list → D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline\Output\V5\qc_patches\filtered_by_rule\rule_filtered_patches.csv

  Stage 2 — SVM on 13,799 patches ...


c:\Users\datainsight\anaconda3\envs\SurGen\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.7.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  SVM results  — white: 9 (0.1%)  tissue: 13,790 (99.9%)

Step A-2 complete.
  Total patches      : 34,435
  White (excluded)   : 20,645  (60.0%)
  Tissue (kept)      : 13,790  (40.0%)
  Merged CSV         → D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline\Output\V5\patch_metadata_merged.csv
  Non-white CSV      → D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline\Output\V5\patch_metadata_nonwhite.csv


,slide_name,label,patch_number,patch_x,patch_y,avg_R,avg_G,avg_B,std_R,std_G,std_B,black_pixel_ratio,white_pixel_ratio,white_label
0,SR1482_40X_HE_T004_02,nonmsih,55,28160,0,196.638214,168.749542,194.371902,29.847233,65.899147,36.092575,0.003906,0.640839,0
1,SR1482_40X_HE_T004_02,nonmsih,56,28672,0,188.368134,149.265274,183.870377,32.662224,70.226860,41.964756,0.003906,0.452343,0
2,SR1482_40X_HE_T004_02,nonmsih,57,29184,0,203.838074,175.305481,202.538696,35.079388,68.348892,38.462173,0.003906,0.594116,0
3,SR1482_40X_HE_T004_02,nonmsih,176,26112,512,200.639999,197.185394,199.524429,50.538925,53.264492,50.463207,0.058594,0.878998,0
4,SR1482_40X_HE_T004_02,nonmsih,177,26624,512,204.137115,181.737976,202.963425,21.231968,54.222290,23.903887,0.000000,0.694283,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13785,SR386_40X_HE_T014_01,nonmsih,20005,58880,59904,238.802002,208.951797,242.470490,22.554190,70.943695,17.009823,0.000000,0.735332,0
13786,SR386_40X_HE_T014_01,nonmsih,20022,67584,59904,212.849548,143.332214,222.237228,26.045595,76.556396,19.819410,0.000000,0.349216,0
13787,SR386_40X_HE_T014_01,nonmsih,20023,68096,59904,228.509888,201.354385,229.835861,17.042259,61.190968,13.226528,0.000000,0.764572,0
13788,SR386_40X_HE_T014_01,nonmsih,20024,68608,59904,211.129211,128.711060,219.758179,16.298309,59.368023,13.324114,0.000000,0.210663,0


## Step QC — Visual patch inspection

> **Controlled by `VISUALIZE_PATCHES` flag** (set in Cell 1):
> - `VISUALIZE_PATCHES = 1` → saves sample PNGs into the QC folders (original behaviour)
> - `VISUALIZE_PATCHES = 0` → skips this step entirely, no PNGs written

Saves sample PNGs into three folders for manual review:

| Folder | Contents |
|---|---|
| `qc_patches/white/` | Patches the pipeline excluded as white/background |
| `qc_patches/nonwhite/` | Patches kept as tissue (fed to Step C) |
| `qc_patches/filtered_by_rule/black_pixel_ratio/` | Excluded by black pixel rule |
| `qc_patches/filtered_by_rule/white_pixel_ratio/` | Excluded by white pixel ratio rule |
| `qc_patches/filtered_by_rule/white_mean_std/` | Excluded by mean+std fallback rule |

In [7]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from pylibCZIrw import czi as pyczi

MAX_PATCHES_PER_CLASS = 200000
MAX_FILTERED_PREVIEW  = 200000

if VISUALIZE_PATCHES == 0:
    print("[Step QC] VISUALIZE_PATCHES=0 — skipping all PNG patch saving.")
else:
    print("[Step QC] VISUALIZE_PATCHES=1 — saving QC patches ...")

    for path, name in [(MERGED_METADATA_CSV, 'Merged'), (NONWHITE_METADATA_CSV, 'Non-white')]:
        if not os.path.exists(path):
            raise FileNotFoundError(f"[ERROR] {name} CSV not found: {path}\n  → Run Step A-2 first.")

    merged_df   = pd.read_csv(MERGED_METADATA_CSV)
    nonwhite_df = pd.read_csv(NONWHITE_METADATA_CSV)

    if 'white_label' not in merged_df.columns:
        raise ValueError("[ERROR] 'white_label' column missing.\n  → Run Step A-2 first.")

    nonwhite_class = merged_df['white_label'].min()
    white_df_all   = merged_df[merged_df['white_label'] != nonwhite_class]
    tissue_df_all  = merged_df[merged_df['white_label'] == nonwhite_class]

    print(f"Total patches  : {len(merged_df):,}")
    print(f"White (excluded): {len(white_df_all):,}")
    print(f"Tissue (kept)  : {len(tissue_df_all):,}")
    print(f"Sampling up to {MAX_PATCHES_PER_CLASS} per class ...\n")

    white_sample  = white_df_all.sample(min(MAX_PATCHES_PER_CLASS, len(white_df_all)),
                                         random_state=42).reset_index(drop=True)
    tissue_sample = tissue_df_all.sample(min(MAX_PATCHES_PER_CLASS, len(tissue_df_all)),
                                          random_state=42).reset_index(drop=True)

    qc_white_dir    = os.path.join(OUTPUT_ROOT, "qc_patches", "white")
    qc_nonwhite_dir = os.path.join(OUTPUT_ROOT, "qc_patches", "nonwhite")
    os.makedirs(qc_white_dir,    exist_ok=True)
    os.makedirs(qc_nonwhite_dir, exist_ok=True)

    czi_index = {Path(p).stem: p for p in find_all_czi_files(CZI_ROOT)}

    def save_patches_to_dir(sample_df: pd.DataFrame, out_dir: str, label_tag: str):
        """Open each CZI once and save all sampled patches from it as PNGs."""
        for slide_stem, group in tqdm(sample_df.groupby('slide_name'),
                                       desc=f"Saving {label_tag}", unit="slide"):
            if slide_stem not in czi_index:
                print(f"  [WARN] CZI not found for '{slide_stem}' — skipping")
                continue
            czi_path = czi_index[slide_stem]
            try:
                slide_info = open_czi_at_target_mag(czi_path, TARGET_MAG, PATCH_SIZE)
            except Exception as e:
                print(f"  [ERROR] Cannot open {slide_stem}: {e}")
                continue

            downsample  = slide_info['downsample']
            x_start     = slide_info['X_start']
            y_start     = slide_info['Y_start']
            zoom        = 1.0 / downsample
            size_native = PATCH_SIZE * downsample

            with pyczi.open_czi(czi_path) as czidoc:
                for _, row in group.iterrows():
                    x_native = x_start + (int(row['patch_x']) * downsample)
                    y_native = y_start + (int(row['patch_y']) * downsample)
                    try:
                        raw = czidoc.read(
                            roi=(x_native, y_native, size_native, size_native),
                            zoom=zoom, plane={'C': 0, 'Z': 0, 'T': 0},
                        )
                        img   = Image.fromarray(raw[:, :, :3].astype('uint8')).convert('RGB')
                        fname = f"{slide_stem}_p{int(row['patch_number'])}.png"
                        img.save(os.path.join(out_dir, fname))
                    except Exception as e:
                        tqdm.write(f"    [ERROR] patch {row['patch_number']} of {slide_stem}: {e}")

    save_patches_to_dir(white_sample,  qc_white_dir,    "white")
    save_patches_to_dir(tissue_sample, qc_nonwhite_dir, "non-white (tissue)")

    print(f"\nWhite QC     → {qc_white_dir}  ({len(white_sample)} patches)")
    print(f"Nonwhite QC  → {qc_nonwhite_dir}  ({len(tissue_sample)} patches)")

    filtered_csv = os.path.join(QC_FILTERED_DIR, "rule_filtered_patches.csv")
    if not os.path.exists(filtered_csv):
        print("\n[SKIP] rule_filtered_patches.csv not found — run Step A-2 first.")
    else:
        filtered_df = pd.read_csv(filtered_csv)
        print(f"\nSaving filtered-by-rule previews ({len(filtered_df):,} total filtered patches) ...")
        for reason, group in filtered_df.groupby('reject_reason'):
            reason_dir = os.path.join(QC_FILTERED_DIR, reason)
            os.makedirs(reason_dir, exist_ok=True)
            sample = group.sample(min(MAX_FILTERED_PREVIEW, len(group)), random_state=42)
            save_patches_to_dir(sample, reason_dir, f"filtered/{reason}")
            print(f"  {reason}: saved {len(sample)} sample patches → {reason_dir}")
        print(f"\nFiltered QC  → {QC_FILTERED_DIR}")
        print(f"  Subfolders : {[d for d in os.listdir(QC_FILTERED_DIR) if os.path.isdir(os.path.join(QC_FILTERED_DIR, d))]}")


[Step QC] VISUALIZE_PATCHES=1 — saving QC patches ...
Total patches  : 34,435
White (excluded): 20,645
Tissue (kept)  : 13,790
Sampling up to 200000 per class ...



Saving white:   0%|          | 0/2 [00:00<?, ?slide/s]

    bbox raw: X=(-397440, -269056), Y=(68036, 185928)


Saving white:  50%|█████     | 1/2 [00:50<00:50, 50.40s/slide]

    bbox raw: X=(-390240, -215648), Y=(76680, 197768)


...

    bbox raw: X=(-397440, -269056), Y=(68036, 185928)


Saving filtered/white_pixel_ratio:  50%|█████     | 1/2 [00:07<00:07,  7.77s/slide]

    bbox raw: X=(-390240, -215648), Y=(76680, 197768)


Saving filtered/white_pixel_ratio: 100%|██████████| 2/2 [00:16<00:00,  8.08s/slide]

  white_pixel_ratio: saved 557 sample patches → D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline\Output\V5\qc_patches\filtered_by_rule\white_pixel_ratio

Filtered QC  → D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline\Output\V5\qc_patches\filtered_by_rule
  Subfolders : ['black_pixel_ratio', 'white_mean_std', 'white_pixel_ratio']


## Step B — Patch extraction helper (runs inline during Step C)

In [8]:
import os
import numpy as np
import pandas as pd
from PIL import Image

# Step B is handled inline during Step C.
# This cell defines the helper that extracts a single patch from the
# already-loaded slide array using the (x, y) coordinates stored in
# the non-white CSV — no PNG files are ever written to disk.

def extract_patch(slide_array: np.ndarray, x: int, y: int,
                  patch_size: int) -> Image.Image:
    """
    Extract a single patch from a slide numpy array at position (x, y).
    Returns a PIL Image in RGB mode of size (patch_size, patch_size).
    """
    patch = slide_array[y : y + patch_size, x : x + patch_size]
    return Image.fromarray(patch.astype(np.uint8)).convert("RGB")


print("Step B helper loaded.  Patches will be extracted in-memory during Step C.")


Step B helper loaded.  Patches will be extracted in-memory during Step C.


## Step C — Feature extraction with H-Optimus-1 (batched GPU inference)

In [9]:
import torch
import timm
from PIL import Image
from torchvision import transforms
from huggingface_hub import login


class HOptimusExtractor:
    """
    Loads H-Optimus-1 and extracts FiveCrop features from patch images.

    extract()       — single patch → (5, 1536)  [kept for compatibility]
    extract_batch() — list of patches → (B, 5, 1536)  [NEW: much faster]
    """

    def __init__(self, device: str = 'cuda', hf_token: str = None):
        if hf_token:
            login(token=hf_token)

        if device == 'cuda' and not torch.cuda.is_available():
            print("  [WARN] CUDA requested but not available — falling back to CPU")
            device = 'cpu'

        self.device = torch.device(device)
        print(f"  Loading H-Optimus-1 on {self.device} ...")
        print(f"  CUDA available: {torch.cuda.is_available()}")
        if torch.cuda.is_available():
            print(f"  GPU: {torch.cuda.get_device_name(0)}")

        self.model = timm.create_model(
            "hf-hub:bioptimus/H-optimus-1",
            pretrained=True,
            init_values=1e-5,
            dynamic_img_size=False,
        ).to(self.device)
        self.model.eval()

        self._crop_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.707223, 0.578729, 0.703617),
                std=(0.211883, 0.230117, 0.177517),
            )
        ])

        self._fivecrop = transforms.Compose([
            transforms.FiveCrop(256),
            transforms.Lambda(
                lambda crops: torch.stack([self._crop_transform(c) for c in crops])
            )
        ])

        print("  H-Optimus-1 ready.")

    def extract(self, image: Image.Image) -> torch.Tensor:
        """Single-patch extract — returns (5, 1536)."""
        crops = self._fivecrop(image.convert("RGB")).to(self.device)
        with torch.autocast(device_type=self.device.type, dtype=torch.float16):
            with torch.inference_mode():
                features = self.model(crops)   # (5, 1536)
        return features.cpu()

    def extract_batch(self, images: list) -> torch.Tensor:
        """
        Batch extract — runs all patches in one GPU forward pass.
        Returns (B, 5, 1536).  Keeps GPU near 100% utilisation.

        Each patch's 5 crops are stacked into a single (B*5, 3, 224, 224) tensor
        so the model sees one large batch instead of B tiny ones.
        """
        batch = torch.cat(
            [self._fivecrop(img.convert("RGB")) for img in images], dim=0
        ).to(self.device)   # (B*5, 3, 224, 224)

        with torch.autocast(device_type=self.device.type, dtype=torch.float16):
            with torch.inference_mode():
                features = self.model(batch)   # (B*5, 1536)

        B = len(images)
        return features.cpu().reshape(B, 5, 1536)   # (B, 5, 1536)


# ── Instantiate extractor (runs once) ─────────────────────────────────────────
extractor = HOptimusExtractor(device=DEVICE, hf_token=HF_TOKEN)


c:\Users\datainsight\anaconda3\envs\SurGen\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Loading H-Optimus-1 on cuda ...
  CUDA available: True
  GPU: NVIDIA GeForce RTX 4090
  H-Optimus-1 ready.


## Step C — Main loop: load slide → extract non-white patches → save features

In [10]:
import os
import torch
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from pylibCZIrw import czi as pyczi


def _read_patch_worker(czidoc, x_native, y_native, size_native, zoom):
    """Read one patch from an open czidoc — called from a background thread."""
    raw = czidoc.read(
        roi=(x_native, y_native, size_native, size_native),
        zoom=zoom, plane={'C': 0, 'Z': 0, 'T': 0},
    )
    return Image.fromarray(raw[:, :, :3].astype('uint8')).convert('RGB')


def run_step_C(czi_root, nonwhite_csv, features_root,
               extractor, patch_size, target_mag, batch_size: int = 8):
    """
    Batched + prefetched feature extraction.

    Speed-ups vs. original one-patch-at-a-time loop:
      1. extract_batch(): FEATURE_BATCH_SIZE patches → one GPU forward pass.
         GPU utilisation goes from ~5% to near 100%.
      2. ThreadPoolExecutor prefetch: while the GPU processes batch N,
         background threads read CZI patches for batch N+1, hiding I/O latency.
      3. Features saved as float16 — halves .pt file size with no quality loss.
    """
    if not os.path.exists(nonwhite_csv):
        raise FileNotFoundError(
            f"[ERROR] Non-white CSV not found: {nonwhite_csv}\n"
            f"  → Run Step A-1 and Step A-2 first."
        )

    nonwhite_df  = pd.read_csv(nonwhite_csv)
    slide_groups = nonwhite_df.groupby('slide_name')
    print(f"Found {len(slide_groups)} slide(s) with non-white patches.")

    czi_index = {Path(p).stem: p for p in find_all_czi_files(czi_root)}

    for slide_stem, group in tqdm(slide_groups, desc="Step C: Feature extraction", unit="slide"):
        feat_dir = os.path.join(features_root, slide_stem)
        os.makedirs(feat_dir, exist_ok=True)

        existing = {Path(f).stem for f in os.listdir(feat_dir) if f.endswith('.pt')}

        if slide_stem not in czi_index:
            print(f"  [WARN] CZI not found for slide '{slide_stem}' — skipping")
            continue

        czi_path = czi_index[slide_stem]
        print(f"  Loading {slide_stem} ...")
        try:
            slide_info = open_czi_at_target_mag(czi_path, target_mag, patch_size)
        except Exception as e:
            print(f"  [ERROR] Cannot open {slide_stem}: {e}")
            continue

        downsample  = slide_info['downsample']
        x_start     = slide_info['X_start']
        y_start     = slide_info['Y_start']
        zoom        = 1.0 / downsample
        size_native = patch_size * downsample

        patches_to_process = group.reset_index(drop=True)
        patches_to_process['_id'] = (
            patches_to_process['slide_name'] + '_'
            + patches_to_process['patch_number'].astype(str)
        )
        patches_to_process = patches_to_process[
            ~patches_to_process['_id'].isin(existing)
        ]

        if patches_to_process.empty:
            tqdm.write(f"  [SKIP] All patches already processed for {slide_stem}")
            continue

        rows      = patches_to_process.to_dict('records')
        n_total   = len(rows)
        n_done    = 0
        n_batches = (n_total + batch_size - 1) // batch_size

        with pyczi.open_czi(czi_path) as czidoc,              ThreadPoolExecutor(max_workers=batch_size) as pool:

            for b_idx in tqdm(range(n_batches),
                               desc=f"  Patches: {slide_stem}",
                               unit="batch", leave=False):

                batch_rows = rows[b_idx * batch_size : (b_idx + 1) * batch_size]
                patch_ids, futures = [], []

                for row in batch_rows:
                    x_native = x_start + (int(row['patch_x']) * downsample)
                    y_native = y_start + (int(row['patch_y']) * downsample)
                    patch_id = f"{slide_stem}_{int(row['patch_number'])}"
                    patch_ids.append(patch_id)
                    futures.append(
                        pool.submit(_read_patch_worker,
                                    czidoc, x_native, y_native, size_native, zoom)
                    )

                # Collect patches (I/O in parallel with previous GPU batch)
                images, valid_ids = [], []
                for patch_id, fut in zip(patch_ids, futures):
                    try:
                        images.append(fut.result())
                        valid_ids.append(patch_id)
                    except Exception as e:
                        tqdm.write(f"    [ERROR] patch {patch_id}: {e}")

                if not images:
                    continue

                # ── GPU batch inference ──────────────────────────────────────
                try:
                    batch_features = extractor.extract_batch(images)   # (B, 5, 1536)
                except Exception as e:
                    tqdm.write(f"    [ERROR] batch inference at idx {b_idx}: {e}")
                    continue

                # ── Save each patch as float16 .pt ───────────────────────────
                for patch_id, feat in zip(valid_ids, batch_features):
                    save_path = os.path.join(feat_dir, patch_id + '.pt')
                    torch.save(feat.to(torch.float16), save_path)
                    n_done += 1

        tqdm.write(
            f"  ✓ {slide_stem}: saved {n_done}/{n_total} feature files → {feat_dir}"
        )


# ── Run Step C ────────────────────────────────────────────────────────────────
run_step_C(
    czi_root      = CZI_ROOT,
    nonwhite_csv  = NONWHITE_METADATA_CSV,
    features_root = FEATURES_ROOT,
    extractor     = extractor,
    patch_size    = PATCH_SIZE,
    target_mag    = TARGET_MAG,
    batch_size    = FEATURE_BATCH_SIZE,
)

print("\nPipeline complete.")
print(f"  Feature files are in : {FEATURES_ROOT}")
print(f"  Non-white patch CSV  : {NONWHITE_METADATA_CSV}")
print(f"  Merged metadata CSV  : {MERGED_METADATA_CSV}")


Found 2 slide(s) with non-white patches.


Step C: Feature extraction:   0%|          | 0/2 [00:00<?, ?slide/s]

  Loading SR1482_40X_HE_T004_02 ...
    bbox raw: X=(-397440, -269056), Y=(68036, 185928)


Step C: Feature extraction:  50%|█████     | 1/2 [00:06<00:06,  6.09s/slide]

  ✓ SR1482_40X_HE_T004_02: saved 137/137 feature files → D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline\Output\V5\features\SR1482_40X_HE_T004_02
  Loading SR386_40X_HE_T014_01 ...
    bbox raw: X=(-390240, -215648), Y=(76680, 197768)


Step C: Feature extraction: 100%|██████████| 2/2 [00:19<00:00,  9.77s/slide]

  ✓ SR386_40X_HE_T014_01: saved 317/317 feature files → D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline\Output\V5\features\SR386_40X_HE_T014_01

Pipeline complete.
  Feature files are in : D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline\Output\V5\features
  Non-white patch CSV  : D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline\Output\V5\patch_metadata_nonwhite.csv
  Merged metadata CSV  : D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline\Output\V5\patch_metadata_merged.csv


## Note — float16 for saved feature tensors

Features are saved as `torch.float16`. Here is an honest assessment:

**Benefits**
- File size is **halved** vs float32 — at shape `(5, 1536)` per patch that is ~15 KB vs ~30 KB. Across hundreds of thousands of patches this is meaningful.
- Loading is faster because less data moves from disk to RAM.

**Downsides — and whether they matter here**
| Concern | Detail | Impact for this use case |
|---|---|---|
| Reduced precision | float16 has ~3 decimal digits vs ~7 for float32 | **Low.** Downstream classifiers (MLP, linear probe) are robust to this level of quantisation noise. The model itself runs in float16 during inference anyway (AMP). |
| Overflow / underflow | float16 range is ±65504; values outside become `inf` or `0` | **Very low.** ViT features from H-Optimus-1 are well within this range. |
| Accumulation errors | Summing many float16 values can drift | **Not applicable.** Features are stored and read individually, not summed. |
| Compatibility | Some older code expects float32 and will error | **Easy to fix.** Cast on load: `feat = torch.load(path).float()` |

**Verdict:** float16 is fine for storing and loading patch features. If you are training a downstream model and want to be safe, cast to float32 after loading with `.float()`. That costs nothing at training time and eliminates any concern.


## Diagnostic — verify saved feature tensor

In [11]:
# Diagnostic — check actual tensor size on disk vs in memory
import os
import torch

# Update this path to any .pt file from your run
SAMPLE_PT = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Complete_Pipeline\Output\V5\features\SR1482_40X_HE_T004_02\SR1482_40X_HE_T004_02_55.pt"

t = torch.load(SAMPLE_PT)
print(f"Loaded dtype : {t.dtype}, shape: {t.shape}")
print(f"Expected bytes: {t.numel() * t.element_size()}")
print(f"Actual file size: {os.path.getsize(SAMPLE_PT)} bytes")


Loaded dtype : torch.float16, shape: torch.Size([5, 1536])
Expected bytes: 15360
Actual file size: 16689 bytes


C:\Users\datainsight\AppData\Local\Temp\ipykernel_28172\3378325236.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  t = torch.load(SAMPLE_PT)
